# CatBoost 모델 구성 설명

## 목표와 Target

- 목표: 학생별로 **해당 예측 주차 종료 시점에서 다음 주 이탈확률**을 예측합니다.
- 사용 데이터: `used_data/weekly_next_week_with_vle.csv`
- Target: `target_next_week_withdrawn` — 다음 7일 안에 이탈하면 1, 아니면 0
- 비교 범위: demo2가 보유한 4·19·25주차

## 사용 Feature

모델 입력은 `id_student`와 Target을 제외한 **98개 Feature**입니다. `id_student`는 모델 입력에 넣지 않고 GroupKFold 분할 기준으로만 사용합니다.

- 과목·운영·주차: `code_module`, `code_presentation`, `prediction_week`, `cutoff_day`, 강좌 운영 길이
- 학생 특성: 성별, 지역, 최종 학력, IMD 구간, 연령대, 이전 수강 횟수, 수강 학점, 장애 여부
- 등록 Feature: 등록 시점, 사전 등록일 수, 지연 등록일 수
- 평가 Feature: 제출·미제출·지각·점수·평가 비중과 현재 주차 기준 누적값
- VLE Feature: 누적 클릭·활동일·사이트·활동 유형별 클릭과 최근 활동 변화량

기존 `target`은 최종 이탈 여부이므로 완전히 제거하고, `date_unregistration`은 다음 주 Target 생성 직후 삭제했습니다. 미래 주차 VLE·평가 정보는 Feature에 포함하지 않습니다.

## CatBoost 전처리·검증

- 범주형 8개는 문자열로 변환한 뒤 CatBoost에 직접 전달합니다.
- `id_student` 기준 **3-Fold GroupKFold**로 동일 학생이 Train과 Test에 섞이지 않게 합니다.
- demo_1과 같은 확률 비교를 위해 이 비교본에는 클래스 가중치를 적용하지 않습니다.

## 하이퍼파라미터

| 항목 | 설정값 |
|---|---:|
| `loss_function` | `Logloss` |
| `eval_metric` | `PRAUC` |
| `iterations` | 500 |
| `learning_rate` | 0.05 |
| `depth` | 7 |
| `l2_leaf_reg` | 5 |
| `od_wait` | 40 |
| `random_seed` | 42 |

## 비교 시 주의

파일·코드·지표 서식은 demo_1과 같지만, demo_1은 1~38주차이고 이 파일은 4·19·25주차입니다. 따라서 현재 결과는 코드 구조와 Feature 구성 비교용이며, 수치의 완전한 A/B 비교에는 같은 주차 행이 필요합니다.


# CatBoost: 주차별 다음 주 이탈 예측

`weekly_next_week_with_vle.csv`를 사용합니다. 아래 셀을 실행하면 demo_1과 같은 이름의 지표·Fold·OOF CSV가 생성됩니다. 코드와 주석을 바꾸려면 `02_catboost_weekly_next_week.py`의 **사용자가 직접 수정할 수 있는 구간**을 편집하세요.


In [ ]:
import runpy
runpy.run_path('02_catboost_weekly_next_week.py', run_name='__main__')

In [ ]:
import pandas as pd
pd.read_csv('catboost_weekly_next_week_metrics.csv')